# 🔴 Interagindo com Redis usando Python

Este notebook é um guia didático e prático que demonstra como interagir com o **Redis** (banco de dados NoSQL do tipo **Chave-Valor**) utilizando a linguagem Python.

## 🛠️ O que é o Redis?
O Redis (Remote Dictionary Server) é um armazenamento de estrutura de dados em memória, usado como banco de dados, cache e message broker. Ele é extremamente rápido porque mantém os dados na memória RAM, persistindo-os em disco de forma assíncrona.

### Detalhes da Conexão Local (Docker Compose):
- **Host:** `localhost`
- **Porta:** `6379`
- **Autenticação:** Nenhuma (configuração padrão local)

## 1. Instalação do Cliente Python
Para nos conectarmos ao Redis, utilizaremos a biblioteca oficial do Python chamada `redis`.

In [ ]:
!pip install redis

## 2. Conectando ao Banco de Dados
Vamos importar a biblioteca e criar uma instância de conexão. 

> **Dica:** O parâmetro `decode_responses=True` converte automaticamente as respostas do Redis (que por padrão vêm como bytes) para strings normais do Python.

In [ ]:
import redis

# Criar conexão com o cliente local Redis
try:
    client = redis.Redis(host='localhost', port=6379, decode_responses=True)
    
    # O comando ping() testa se a conexão está funcionando
    if client.ping():
        print("✅ Conexão com Redis estabelecida com sucesso!")
except Exception as e:
    print(f"❌ Erro ao conectar ao Redis: {e}")
    print("Certifique-se de que o container do Redis está rodando (use 'make up' ou 'docker compose up -d')")

## 3. Operações CRUD Básicas (Chaves do Tipo String)
O tipo mais básico de valor no Redis é a **String** (que pode conter texto, números ou até binários serializados).

In [ ]:
# === CREATE (Inserir) ===
# Define o valor da chave 'usuario:1:nome'
client.set('usuario:1:nome', 'Carlos Silva')
print("✍️ Nome inserido.")

# === READ (Ler / Coletar) ===
nome_usuario = client.get('usuario:1:nome')
print(f"📖 Valor recuperado para 'usuario:1:nome': {nome_usuario}")

# === UPDATE (Atualizar) ===
# No Redis, um novo SET na mesma chave simplesmente sobrescreve o valor antigo
client.set('usuario:1:nome', 'Carlos Souza')
nome_atualizado = client.get('usuario:1:nome')
print(f"🔄 Valor atualizado para 'usuario:1:nome': {nome_atualizado}")

# === DELETE (Deletar) ===
# Remove a chave do banco
client.delete('usuario:1:nome')
print("🗑️ Chave deletada.")

# Verificar se a chave ainda existe
existe = client.exists('usuario:1:nome')
print(f"❓ A chave 'usuario:1:nome' existe? {'Sim' if existe else 'Não'}")

## 4. Controle de Expiração (TTL - Time to Live)
Uma das funcionalidades mais poderosas do Redis é a capacidade de definir uma data/tempo de expiração automática para as chaves, muito utilizada para sessões e caches temporários.

In [ ]:
import time

# Inserir uma chave que expira em 5 segundos usando setex (SET com Expiration)
client.setex('sessao:token', 5, 'jwt_token_exemplo_123')
print("🔑 Token de sessão inserido com expiração de 5 segundos.")

# Consultar o tempo de vida restante (TTL em segundos)
ttl_inicial = client.ttl('sessao:token')
print(f"⏳ TTL inicial: {ttl_inicial} segundos")

# Aguardar 2 segundos
time.sleep(2)
ttl_restante = client.ttl('sessao:token')
valor_token = client.get('sessao:token')
print(f"⏳ TTL após 2 segundos: {ttl_restante} segundos (Valor: {valor_token})")

# Aguardar mais 4 segundos (totalizando 6, estourando os 5 segundos)
print("Aguardando a chave expirar...")
time.sleep(4)
valor_expirado = client.get('sessao:token')
print(f"🗑️ Valor recuperado após expiração: {valor_expirado} (Chave expirou automaticamente!)")

## 5. Estruturas de Dados Avançadas
O Redis suporta várias estruturas de dados além das strings básicas. Vamos explorar **Hashes**, **Lists** e **Sets**.

### A. Hashes (Dicionários/Objetos)
Hashes são ótimos para representar objetos estruturados, contendo múltiplos campos e valores dentro de uma única chave principal.

In [ ]:
chave_hash = 'usuario:100'

# Inserir dados no Hash
client.hset(chave_hash, mapping={
    'nome': 'Alice Silva',
    'email': 'alice@email.com',
    'idade': '28',
    'cidade': 'João Pessoa'
})
print(f"📝 Hash criado em '{chave_hash}'")

# Obter um campo específico
nome = client.hget(chave_hash, 'nome')
print(f"👤 Nome do usuário: {nome}")

# Obter todo o objeto (dicionário)
dados_usuario = client.hgetall(chave_hash)
print(f"📦 Objeto completo: {dados_usuario}")

# Atualizar um único campo do Hash
client.hset(chave_hash, 'idade', '29')
idade_atualizada = client.hget(chave_hash, 'idade')
print(f"🔄 Idade atualizada para: {idade_atualizada}")

# Remover um campo do Hash
client.hdel(chave_hash, 'cidade')
dados_finais = client.hgetall(chave_hash)
print(f"🗑️ Hash após deletar o campo 'cidade': {dados_finais}")

# Limpar chave para manter ambiente organizado
client.delete(chave_hash)

### B. Listas (Fila / Pilha)
Listas do Redis são coleções de strings ordenadas pela ordem de inserção. Você pode adicionar elementos no início (`lpush`) ou no fim (`rpush`).

In [ ]:
chave_lista = 'tarefas:urgentes'
client.delete(chave_lista) # Limpar se já existir

# Empurrar itens para o fim da lista (Right Push)
client.rpush(chave_lista, 'Enviar e-mail para o cliente')
client.rpush(chave_lista, 'Revisar PR pendente')
# Empurrar item para o início da lista (Left Push)
client.lpush(chave_lista, 'Corrigir bug crítico em produção')

# Obter tamanho da lista
tamanho = client.llen(chave_lista)
print(f"📋 Tamanho da lista de tarefas: {tamanho}")

# Coletar todos os itens da lista (índice 0 até -1)
itens = client.lrange(chave_lista, 0, -1)
print(f"🔍 Lista completa de tarefas: {itens}")

# Consumir/Remover o primeiro item (Left Pop)
primeira_tarefa = client.lpop(chave_lista)
print(f"✅ Tarefa concluída e removida da fila: '{primeira_tarefa}'")

# Lista restante
itens_restantes = client.lrange(chave_lista, 0, -1)
print(f"🔍 Lista restante: {itens_restantes}")

# Limpar chave
client.delete(chave_lista)

### C. Sets (Conjuntos Únicos e Não Ordenados)
Sets são coleções de strings únicas (sem duplicidade). Útil para tags, listas de presença, seguidores, etc.

In [ ]:
chave_set = 'tags:post:42'
client.delete(chave_set) # Limpar se já existir

# Adicionar itens ao Set
client.sadd(chave_set, 'tecnologia')
client.sadd(chave_set, 'programacao')
client.sadd(chave_set, 'nosql')
# Tentar adicionar item duplicado
client.sadd(chave_set, 'nosql')

# Coletar todos os membros do Set
membros = client.smembers(chave_set)
print(f"🏷️ Tags do Post (repare que 'nosql' aparece apenas uma vez): {membros}")

# Verificar se uma tag específica está no Set
tem_python = client.sismember(chave_set, 'python')
tem_nosql = client.sismember(chave_set, 'nosql')
print(f"❓ Contém a tag 'python'? {'Sim' if tem_python else 'Não'}")
print(f"❓ Contém a tag 'nosql'? {'Sim' if tem_nosql else 'Não'}")

# Remover um elemento do Set
client.srem(chave_set, 'tecnologia')
membros_finais = client.smembers(chave_set)
print(f"🗑️ Tags restantes após remover 'tecnologia': {membros_finais}")

# Limpar chave
client.delete(chave_set)

## 🏁 Conclusão
Parabéns! Você concluiu a introdução ao Redis. Você aprendeu a:
- Conectar ao Redis em Python utilizando a biblioteca `redis`.
- Salvar e obter strings com controle de expiração (TTL).
- Manipular dados estruturados em Dicionários (Hashes), Filas/Pilhas (Lists) e Conjuntos sem duplicidade (Sets).

Para continuar estudando, experimente criar rotinas de cache mais complexas ou testar transações e operações atômicas com `client.pipeline()`!